# 03a · RAG question set (no API key needed)

The question set is fixed and lives in `03_rag/questions_curated.json`, built by `03_rag/make_questions.py`:
**54 English questions** a coffee grower in Karnataka would ask (leaf rust 11, Cercospora 5, leaf miner 8, other pests
and diseases 12, cultivation practice 18) and **21 Kannada versions** of some of them (used to test retrieval across
languages). Every question has a short gold answer and gold evidence quotes copied word-for-word from a knowledge-base
PDF page, mostly the Central Coffee Research Institute's *Coffee Cultivation Guide* (Chikmagalur).

This notebook copies the 14 KB PDFs from the app (read-only), re-checks every evidence quote against the PDF text,
and freezes `03_rag/data/questions.jsonl` (+ SHA-256). Phoma is not in the set: the only Phoma document in the KB
(`Coffee_Diseases.pdf`) extracts as garbled text, so no system can retrieve it (reported as a KB gap).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "03_rag"))

import hashlib
import json
import shutil

import pandas as pd

from common import config, env

config.ensure_dirs()

In [ ]:
for p in sorted(config.APP_KB_DIR.glob("*.pdf")):
    if not (config.KB_DIR / p.name).exists():
        shutil.copy2(p, config.KB_DIR / p.name)
manifest = pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size,
                          "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(config.KB_DIR.glob("*.pdf"))])
manifest.to_csv(config.KB_DIR / "kb_manifest.csv", index=False)
print(len(manifest), "PDFs in the knowledge base")

In [ ]:
import make_questions  # re-verifies every evidence quote against the PDF text (stops if one is missing)

rows = make_questions.build()
if config.SMOKE:
    keep = {r["id"] for r in rows if r["lang"] == "en"}
    keep = set(sorted(keep)[::9])
    rows = [r for r in rows if r["id"] in keep or r.get("pair") in keep]
frozen = config.RAG_DATA / "questions.jsonl"
with frozen.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
digest = hashlib.sha256(frozen.read_bytes()).hexdigest()
(config.RAG_DATA / "questions.jsonl.sha256").write_text(f"{digest}  questions.jsonl\n")
qs = pd.DataFrame(rows)
env.write_json(config.RAG_DATA / "questions_meta.json", {
    "n": len(qs), "english": int((qs.lang == "en").sum()), "kannada": int((qs.lang == "kn").sum()),
    "selection": "written by the authors from the knowledge base; every evidence quote verified against the PDF text",
    "sha256": digest, "meta": env.run_meta()})
print(f"FROZEN {len(qs)} questions -> {frozen.name}  sha256 {digest[:16]}...")
qs.groupby(["topic", "lang"]).size().unstack(fill_value=0)